[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crunchdao/crunch-numinous/blob/main/numinous/examples/quickstart.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/numinous/assets/banner.webp)

# Numinous — Binary Event Forecasting

Predict the probability that real-world events resolve **"Yes"**.

Events are sourced from prediction markets and cover politics, crypto, sports, weather, and more.

## Scoring

Predictions are evaluated with the **Brier score**:

$$\text{Brier} = (p - o)^2$$

where $p$ is your predicted probability and $o \in \{0, 1\}$ is the actual outcome. **Lower is better.**

| Score | Meaning |
|-------|---------|
| 0.00  | Perfect |
| 0.25  | Uninformative (always 0.5) |
| 1.00  | Worst possible |

Predictions are clipped to $[0.01, 0.99]$. Missing predictions are scored as 0.25.

## Setup

In [ ]:
%pip install crunch-numinous --upgrade --quiet --progress-bar off
!crunch-numinous --version

## Gateway

Your notebook has **no direct internet access** in production. All external calls (LLMs, search, OSINT...) go through the **gateway**.

- **In production**: `SANDBOX_PROXY_URL` is set automatically — API costs are covered by Crunch
- **Locally**: you run the gateway yourself with your own API keys — most providers offer a free tier

See the [API Reference](https://github.com/crunchdao/crunch-numinous/blob/main/numinous/gateway/API_REFERENCE.md) for all available endpoints.

In [ ]:
import os

# @crunch/keep:on
GATEWAY_URL = os.environ.get("SANDBOX_PROXY_URL", "http://localhost:8090")

### Configure an API Key

#### Option 1: Specify API Keys

For maximum security, specify your API keys in a secure but volatile manner. <br />
They will not be saved locally and will never be sent to Crunch.

**Note:** If you restart your Kernel, you have to enter them again!

In [ ]:
# @crunch/keep:none
from getpass import getpass

# Get an API Key here: https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ") or None

# Get an API Key here: https://openrouter.ai/settings/keys
# os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ") or None

# Get an API Key here: https://chutes.ai/app
# os.environ["CHUTES_API_KEY"] = getpass("Enter your Chutes API key: ") or None

#### Option 2: Use a .env

Use the same keys (`OPENAI_API_KEY`, `OPENROUTER_API_KEY` or `CHUTES_API_KEY`) as above in the file located here:

In [ ]:
# @crunch/keep:none
from pathlib import Path

print(f".env file location: {Path.home() / '.crunch-numinous-gateway.env'}")

#### Option 3: Configure using the CLI

Run the following command in your terminal:
```bash
crunch-numinous gateway configure
```

**Note:** This option cannot be used directly from a notebook; a terminal must be used instead!

### Run the Gateway locally

In [ ]:
!crunch-numinous gateway restart &

# To see HTTP requests and responses in the logs, use:
# !crunch-numinous gateway restart --debug &

# Then check the logs with:
# !crunch-numinous gateway logs --no-follow

## Your Model

Subclass `TrackerBase` and implement `_predict()`.

```
Input:  {"event_id": "...", "event_type": "llm", "title": "Will X happen?", "description": "...", "cutoff": "...", "metadata": {...}}
Output: {"event_id": "...", "prediction": 0.72, "reasoning": "Based on..."}
```

This example asks an LLM to estimate P(Yes) given the event details. <br />
It calls the gateway's OpenAI endpoint — swap the model or provider to experiment.

In [ ]:
import re

import httpx

from numinous.tracker import TrackerBase


class LLMTracker(TrackerBase):
    """Asks an LLM to estimate P(Yes) for each event."""

    def _predict(self, subject):
        data = self._get_data(subject)
        if not isinstance(data, dict):
            return {"event_id": subject, "prediction": 0.5}

        event_id = data.get("event_id", subject)
        title = data.get("title") or ""
        description = data.get("description") or ""

        prompt = (
            "You are a forecasting expert. Estimate the probability that "
            "this event resolves Yes.\n\n"
            f"Event: {title}\n"
            f"Description: {description}\n\n"
            "Reply with a decimal number between 0 and 1, followed by a "
            "one-sentence explanation. Format: NUMBER | REASON"
        )

        try:
            resp = httpx.post(
                f"{GATEWAY_URL}/api/gateway/openai/responses",
                json={
                    "model": "gpt-5-mini",
                    "input": [{"role": "user", "content": prompt}],
                    "max_output_tokens": 512,
                },
                timeout=180.0,
            )
            resp.raise_for_status()
            result = resp.json()

            text = ""
            for item in result.get("output", []):
                for content in (item.get("content") or []):
                    if content.get("text"):
                        text = content["text"]

            match = re.search(r"(0\.\d+|1\.0|0|1)", text)
            prediction = float(match.group(1)) if match else 0.5
            reasoning = text.split("|", 1)[1].strip() if "|" in text else text
        except Exception as e:
            print(f"  [{event_id}] LLM call failed: {e}, falling back to 0.5")
            prediction = 0.5
            reasoning = None

        return {
            "event_id": event_id,
            "prediction": max(0.0, min(1.0, prediction)),
            "reasoning": reasoning,
        }

## Test Locally

Feed sample events and check predictions.

In [3]:
SAMPLE_EVENTS = [
    {"event_id": "evt-btc-100k", "event_type": "crypto", "title": "Will BTC exceed $100k by end of March?",
     "cutoff": "2026-04-01T00:00:00Z", "description": "Resolves YES if BTC/USD > $100,000.",
     "metadata": {"market_type": "Crypto", "topics": ["Crypto"]}},
    {"event_id": "evt-rain-nyc", "event_type": "weather", "title": "Will it rain in NYC tomorrow?",
     "cutoff": "2026-03-25T00:00:00Z", "description": "Resolves YES if measurable precipitation recorded.",
     "metadata": {"market_type": "Weather", "topics": ["Weather"]}},
    {"event_id": "evt-fed-rate", "event_type": "finance", "title": "Will the Fed cut rates in April?",
     "cutoff": "2026-04-30T00:00:00Z", "description": "Resolves YES if FOMC announces a rate cut.",
     "metadata": {"market_type": "Finance", "topics": ["Finance"]}},
]

OUTCOMES = {"evt-btc-100k": 1, "evt-rain-nyc": 1, "evt-fed-rate": 0}

In [ ]:
# @crunch/keep:none

from rich.console import Console
from rich.table import Table
from tqdm.auto import tqdm

from numinous.scoring import EventGroundTruth, ForecastOutput, score_prediction

model = LLMTracker()
scores = []

for event in tqdm(SAMPLE_EVENTS):
    model.feed_update(event)
    result = model.predict(event["event_id"])

    score = score_prediction(
        ForecastOutput(event_id=event["event_id"], prediction=result["prediction"]),
        EventGroundTruth(event_id=event["event_id"], outcome=OUTCOMES[event["event_id"]]),
    )

    scores.append((event, score))

table = Table(title="Scoring Results")
for name in ("Event", "LLM", "Outcome", "Brier"):
    table.add_column(name, justify="right" if name != "Event" else "left")

for event, score in scores:
    table.add_row(
        event["title"][:50],
        f"{score.clipped_prediction:.3f}",
        "Yes" if score.outcome else "No",
        f"{score.brier_score:.4f}"
    )

avg = sum(score.brier_score for _, score in scores) / len(scores)
table.add_section()
table.add_row("[bold]Average Brier[/bold]", "", "", f"[bold]{avg:.4f}[/bold]")
table.add_row("[dim]Baseline (always 0.5)[/dim]", "", "", "[dim]0.2500[/dim]")

Console().print(table)

## Debugging

If you encounter any issues, you can view the gateway logs, which are useful for diagnosing API errors.

In [ ]:
!crunch-numinous gateway logs --no-follow

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Deploy it in real condition

### >> https://hub.crunchdao.com/competitions/numinous/submit/notebook

The platform will find your `TrackerBase` subclass automatically.

![Download and Submit Notebook](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/download-and-submit-notebook-deployment.gif)